In [1]:
# Uncomment and run this cell if you're on Colab or Kaggle
# !git clone https://github.com/nlp-with-transformers/notebooks.git
# %cd notebooks
# from install import *
# install_requirements()

In [2]:
#hide
from utils import * # importa todo lo de utils.py
setup_chapter()

/home/srmjf/transf/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using transformers v4.57.1
Using datasets v5.0.0


# Hello Transformers

<img alt="transformer-timeline" caption="The transformers timeline" src="images/chapter01_timeline.png" id="transformer-timeline"/>

## The Encoder-Decoder Framework

<img alt="rnn" caption="Unrolling an RNN in time." src="images/chapter01_rnn.png" id="rnn"/>

<img alt="enc-dec" caption="Encoder-decoder architecture with a pair of RNNs. In general, there are many more recurrent layers than those shown." src="images/chapter01_enc-dec.png" id="enc-dec"/>

## Attention Mechanisms

<img alt="enc-dec-attn" caption="Encoder-decoder architecture with an attention mechanism for a pair of RNNs." src="images/chapter01_enc-dec-attn.png" id="enc-dec-attn"/> 

<img alt="attention-alignment" width="500" caption="RNN encoder-decoder alignment of words in English and the generated translation in French (courtesy of Dzmitry Bahdanau)." src="images/chapter02_attention-alignment.png" id="attention-alignment"/> 

<img alt="transformer-self-attn" caption="Encoder-decoder architecture of the original Transformer." src="images/chapter01_self-attention.png" id="transformer-self-attn"/> 

## Transfer Learning in NLP

<img alt="transfer-learning" caption="Comparison of traditional supervised learning (left) and transfer learning (right)." src="images/chapter01_transfer-learning.png" id="transfer-learning"/>  

<img alt="ulmfit" width="500" caption="The ULMFiT process (courtesy of Jeremy Howard)." src="images/chapter01_ulmfit.png" id="ulmfit"/>

## Hugging Face Transformers: Bridging the Gap

## A Tour of Transformer Applications

In [3]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""

### Text Classification

In [4]:
#hide_output
from transformers import pipeline

classifier = pipeline("text-classification")

In [5]:
import pandas as pd

outputs = classifier(text)
pd.DataFrame(outputs)    

,label,score
0,NEGATIVE,0.901545


In [6]:
# Para ver el modelo que ha descargado pipeline()

print(classifier.model.name_or_path)


distilbert/distilbert-base-uncased-finetuned-sst-2-english


In [7]:
# sst-2 -> Stanford Sentiment Treebank 2

# Es uno de los datasets clásicos de análisis de sentimiento en NLP.

Cuando haces:

pipeline("text-classification")

ocurren varias cosas:

1. Detectar la tarea
↓
2. Elegir un modelo por defecto
↓
3. Descargar pesos
↓
4. Descargar tokenizer
↓
5. Construir pipeline

Podrías hacerlo explícitamente:

classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

Y obtendrías el mismo comportamiento.

Muchas veces el flujo será:

Modelo HF

↓

Lista de diccionarios

↓

DataFrame Pandas

↓

Análisis


Por eso los autores dicen que es cómodo usar Pandas.

No porque Pandas exija diccionarios, sino porque las salidas de Hugging Face suelen tener una estructura que Pandas puede convertir en tabla de forma casi automática.

### Named Entity Recognition

In [8]:
ner_tagger = pipeline("ner", aggregation_strategy="simple")
outputs = ner_tagger(text)
pd.DataFrame(outputs)    

,entity_group,score,word,start,end
0,ORG,0.879010,Amazon,5,11
1,MISC,0.990859,Optimus Prime,36,49
2,LOC,0.999755,Germany,90,97
3,MISC,0.556568,Mega,208,212
4,PER,0.590257,##tron,212,216
5,ORG,0.669692,Decept,253,259
6,MISC,0.498350,##icons,259,264
7,MISC,0.775361,Megatron,350,358
8,MISC,0.987854,Optimus Prime,367,380
9,PER,0.812096,Bumblebee,502,511


In [9]:
# Para verlo como listas de diccionarios

outputs


[{'entity_group': 'ORG',
  'score': np.float32(0.87900966),
  'word': 'Amazon',
  'start': 5,
  'end': 11},
 {'entity_group': 'MISC',
  'score': np.float32(0.9908588),
  'word': 'Optimus Prime',
  'start': 36,
  'end': 49},
 {'entity_group': 'LOC',
  'score': np.float32(0.9997547),
  'word': 'Germany',
  'start': 90,
  'end': 97},
 {'entity_group': 'MISC',
  'score': np.float32(0.5565682),
  'word': 'Mega',
  'start': 208,
  'end': 212},
 {'entity_group': 'PER',
  'score': np.float32(0.59025675),
  'word': '##tron',
  'start': 212,
  'end': 216},
 {'entity_group': 'ORG',
  'score': np.float32(0.6696924),
  'word': 'Decept',
  'start': 253,
  'end': 259},
 {'entity_group': 'MISC',
  'score': np.float32(0.49834952),
  'word': '##icons',
  'start': 259,
  'end': 264},
 {'entity_group': 'MISC',
  'score': np.float32(0.7753611),
  'word': 'Megatron',
  'start': 350,
  'end': 358},
 {'entity_group': 'MISC',
  'score': np.float32(0.9878539),
  'word': 'Optimus Prime',
  'start': 367,
  'end':

### Question Answering 

In [10]:
reader = pipeline("question-answering")
question = "What does the customer want?"
outputs = reader(question=question, context=text)
pd.DataFrame([outputs])    

,score,start,end,answer
0,0.631292,335,358,an exchange of Megatron


In [11]:
# el modelo que usamos ahora
print(reader.model.name_or_path)

distilbert/distilbert-base-cased-distilled-squad


In [12]:
# squad -> Fine-tuning sobre Stanford Question Answering Dataset

### Summarization

In [13]:
summarizer = pipeline("summarization")
outputs = summarizer(text, max_length=45, clean_up_tokenization_spaces=True)
print(outputs[0]['summary_text'])

/home/srmjf/transf/lib/python3.12/site-packages/transformers/generation/utils.py:1633: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (45). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(


 Bumblebee ordered an Optimus Prime action figure from your online store in
Germany. Unfortunately, when I opened the package, I discovered to my horror
that I had been sent an action figure of Megatron instead.


In [15]:
# el modelo que usamos ahora
print(summarizer.model.name_or_path)

sshleifer/distilbart-cnn-12-6


In [ ]:
# Éste es un modelo que usa ecoder + decoder. Versión destilada de Bart
# se usó un dataset con miles de artículos de cnn y Daily Mail

### Translation

In [17]:
# Aquí le decimos qué modelo queremos. No lo elige HF según la tarea
translator = pipeline("translation_en_to_de", 
                      model="Helsinki-NLP/opus-mt-en-de")
outputs = translator(text, clean_up_tokenization_spaces=True, min_length=100)
print(outputs[0]['translation_text'])

Sehr geehrter Amazon, letzte Woche habe ich eine Optimus Prime Action Figur aus
Ihrem Online-Shop in Deutschland bestellt. Leider, als ich das Paket öffnete,
entdeckte ich zu meinem Entsetzen, dass ich stattdessen eine Action Figur von
Megatron geschickt worden war! Als lebenslanger Feind der Decepticons, Ich
hoffe, Sie können mein Dilemma verstehen. Um das Problem zu lösen, Ich fordere
einen Austausch von Megatron für die Optimus Prime Figur habe ich bestellt.
Eingeschlossen sind Kopien meiner Aufzeichnungen über diesen Kauf. Ich erwarte,
von Ihnen bald zu hören. Aufrichtig, Bumblebee.


### Text Generation

In [18]:
#hide
from transformers import set_seed
set_seed(42) # Set the seed to get reproducible results

In [22]:
generator = pipeline("text-generation")
response = "Dear Bumblebee, I am sorry to hear that your order was mixed up."
prompt = text + "\n\nCustomer service response:\n" + response
outputs = generator(prompt, max_length=200)
print(outputs[0]['generated_text'])

Dear Amazon, last week I ordered an Optimus Prime action figure from your online
store in Germany. Unfortunately, when I opened the package, I discovered to my
horror that I had been sent an action figure of Megatron instead! As a lifelong
enemy of the Decepticons, I hope you can understand my dilemma. To resolve the
issue, I demand an exchange of Megatron for the Optimus Prime figure I ordered.
Enclosed are copies of my records concerning this purchase. I expect to hear
from you soon. Sincerely, Bumblebee.

Customer service response:
Dear Bumblebee, I am sorry to hear that your order was mixed up. The order was
not correct, and I would like to know why. I received the Optimus Prime action
figure from your online store on November 13, 2013. Please inform me that you
received the toy at a different time and place. I will be contacting you once I
have received the correct order number.

I have received no responses to my inquiry.

Bumblebee's response to your inquiry:

Dear Bumblebee, I 

In [ ]:
# Si sólo quiero lo generado

generated = outputs[0]["generated_text"]

print(generated[len(prompt):])

## The Hugging Face Ecosystem

<img alt="ecosystem" width="500" caption="An overview of the Hugging Face ecosystem of libraries and the Hub." src="images/chapter01_hf-ecosystem.png" id="ecosystem"/>

### The Hugging Face Hub

<img alt="hub-overview" width="1000" caption="The models page of the Hugging Face Hub, showing filters on the left and a list of models on the right." src="images/chapter01_hub-overview.png" id="hub-overview"/> 

<img alt="hub-model-card" width="1000" caption="A example model card from the Hugging Face Hub. The inference widget is shown on the right, where you can interact with the model." src="images/chapter01_hub-model-card.png" id="hub-model-card"/> 

### Hugging Face Tokenizers

### Hugging Face Datasets

### Hugging Face Accelerate

## Main Challenges with Transformers

## Conclusion